# FlyWire synapses -> K cubes (preprocessing for death model v3)

Runs on Google Colab, so the 9.5 GB synapse table is downloaded by Google, not by your own internet.

**What it makes:** for every proofread neuron, how many of its synapses fall in each 10 µm cube of the brain,
separately for output synapses (where it releases) and input synapses (where it receives).
The result is one small file saved to your Google Drive (`flywire_v3/synapse_cubes_10um.npz`).

**A copy of the 9.5 GB file is kept in Drive** (`flywire_v3/raw/`), so later sessions (other cube sizes, other analyses)
copy it from Drive instead of downloading it again from Zenodo.

Run the cells top to bottom (Runtime → Run all). First time: roughly 5–15 min download, a few minutes to copy it to Drive,
and a few minutes of counting.
Colab's free RAM (~12 GB) is enough: only the needed columns are loaded.

In [ ]:
# 1. Connect Google Drive and make an output folder
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/flywire_v3'
os.makedirs(OUT, exist_ok=True)
print('results will be saved in', OUT)

In [ ]:
# 2. Get the synapse table and the list of proofread neurons
#    First time: download from Zenodo to Colab's disk, then keep a copy in Drive.
#    Later sessions: copy it back from Drive (no download from Zenodo needed).
import shutil, subprocess
RAW = f'{OUT}/raw'
os.makedirs(RAW, exist_ok=True)
files = {
    'flywire_synapses_783.feather': 'https://zenodo.org/records/10676866/files/flywire_synapses_783.feather?download=1',
    'proofread_root_ids_783.npy':   'https://zenodo.org/records/10676866/files/proofread_root_ids_783.npy?download=1',
}
for name, url in files.items():
    local, saved = f'/content/{name}', f'{RAW}/{name}'
    if os.path.exists(local):
        print(name, 'already on Colab disk')
    elif os.path.exists(saved):
        print(name, 'found in Drive, copying to Colab disk ...')
        shutil.copy(saved, local + '.part'); os.replace(local + '.part', local)
    else:
        print(name, 'downloading from Zenodo ...')
        # '.part' until complete, so an interrupted download is resumed, never mistaken for a finished one
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', local + '.part', url], check=True)
        os.replace(local + '.part', local)
    if not os.path.exists(saved):
        print('   saving a copy to Drive (takes a few minutes for the big file) ...')
        shutil.copy(local, saved + '.part'); os.replace(saved + '.part', saved)
    print(f'   {name}: {os.path.getsize(local)/1e9:.2f} GB')

In [ ]:
# 3. Settings
CUBE_UM = 10          # cube size in micrometres (K is tracked per cube in model v3)
SLICE   = 5_000_000   # synapses processed at a time (keeps memory low)
POS     = 'pre'       # which synapse point gives the location: 'pre' or 'post' (they are < 1 µm apart)

In [ ]:
# 4. Load only the columns we need
import numpy as np, pyarrow.feather as feather, pyarrow.compute as pc, time
path = '/content/flywire_synapses_783.feather'
cols = ['pre_pt_root_id', 'post_pt_root_id',
        f'{POS}_pt_position_x', f'{POS}_pt_position_y', f'{POS}_pt_position_z']
t0 = time.time()
table = feather.read_table(path, columns=cols)
print(f'{table.num_rows:,} synapses loaded in {time.time()-t0:.0f} s')
print(table.schema)

ids = np.sort(np.load('/content/proofread_root_ids_783.npy').astype(np.int64))
print(f'{len(ids):,} proofread neurons')

In [ ]:
# 5. Grid covering all synapses (positions are in nanometres)
S = CUBE_UM * 1000
lo, hi = [], []
for a in 'xyz':
    mm = pc.min_max(table.column(f'{POS}_pt_position_{a}')).as_py()
    lo.append(mm['min']); hi.append(mm['max'])
lo = np.array(lo, dtype=np.int64); hi = np.array(hi, dtype=np.int64)
dims = (hi - lo) // S + 1
n_cubes = int(np.prod(dims))
print('extent (µm):', (hi - lo) / 1000, ' grid:', dims, f'= {n_cubes:,} cubes')

In [ ]:
# 6. Count synapses per (neuron, cube), for outputs (pre) and inputs (post)
def to_idx(root):
    '''position of each root id in the proofread list, -1 if not proofread'''
    k = np.searchsorted(ids, root)
    k = np.clip(k, 0, len(ids) - 1)
    return np.where(ids[k] == root, k, -1)

def reduce(keys, counts):
    u, inv = np.unique(keys, return_inverse=True)
    return u, np.bincount(inv, weights=counts).astype(np.int64)

acc = {'pre': [], 'post': []}
t0 = time.time()
for start in range(0, table.num_rows, SLICE):
    part = table.slice(start, SLICE)
    xyz = np.stack([part.column(f'{POS}_pt_position_{a}').to_numpy() for a in 'xyz'], 1).astype(np.int64)
    c = (xyz - lo) // S
    cube = (c[:, 0] + dims[0] * (c[:, 1] + dims[1] * c[:, 2])).astype(np.int64)
    for side in ['pre', 'post']:
        n = to_idx(part.column(f'{side}_pt_root_id').to_numpy().astype(np.int64))
        ok = n >= 0
        keys = n[ok] * n_cubes + cube[ok]
        u, cnt = np.unique(keys, return_counts=True)
        acc[side].append((u, cnt))
        if len(acc[side]) >= 8:                      # merge now and then to save memory
            k = np.concatenate([a for a, _ in acc[side]]); w = np.concatenate([b for _, b in acc[side]])
            acc[side] = [reduce(k, w)]
    done = min(start + SLICE, table.num_rows)
    print(f'{done:,} / {table.num_rows:,} synapses  ({time.time()-t0:.0f} s)', end='\r')

result = {}
for side in ['pre', 'post']:
    k = np.concatenate([a for a, _ in acc[side]]); w = np.concatenate([b for _, b in acc[side]])
    u, cnt = reduce(k, w)
    result[side] = (u // n_cubes, u % n_cubes, cnt)
    print(f'\n{side}: {len(u):,} (neuron, cube) pairs, {cnt.sum():,} synapses of proofread neurons')

In [ ]:
# 7. Save the small result to Drive
fname = f'{OUT}/synapse_cubes_{CUBE_UM}um.npz'
np.savez_compressed(
    fname,
    root_ids=ids, cube_um=CUBE_UM, origin_nm=lo, dims=dims, position=POS,
    pre_neuron=result['pre'][0].astype(np.int32),  pre_cube=result['pre'][1].astype(np.int32),
    pre_count=result['pre'][2].astype(np.int32),
    post_neuron=result['post'][0].astype(np.int32), post_cube=result['post'][1].astype(np.int32),
    post_count=result['post'][2].astype(np.int32))
print('saved', fname, f'({os.path.getsize(fname)/1e6:.0f} MB)')

In [ ]:
# 8. Quick look: how many cubes hold synapses, and how spread out neurons are
occupied = np.unique(np.concatenate([result['pre'][1], result['post'][1]]))
print(f'{len(occupied):,} of {n_cubes:,} cubes contain synapses ({len(occupied)*CUBE_UM**3/1e9:.3f} mm^3 of neuropil)')
per_neuron = np.bincount(result['pre'][0], minlength=len(ids)) + np.bincount(result['post'][0], minlength=len(ids))
print('cubes touched per neuron: median', int(np.median(per_neuron)), ' 90th percentile', int(np.percentile(per_neuron, 90)))